# Phase 1: Data Audit & Contracts

## Objective
Perform comprehensive data quality assessment and establish validation contracts.

## Dataset Review
- **Source**: Kaggle Credit Card Fraud Detection
- **Expected Shape**: 284,807 transactions x 31 features
- **Target**: Class (0 = legitimate, 1 = fraud)
- **Expected Imbalance**: ~0.172% fraud rate

In [ ]:
# Import require libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Import our utility functions
from utils import (
    load_fraud_data,
    quick_data_summary,
    plot_target_distribution,
    DATA_RAW
)

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print("=" * 60)
print("PHASE 1: DATA AUDIT & CONTRACTS")
print("=" * 60)

print(f"\n📁 Data directory: {DATA_RAW}")
print("✅ Environment ready\n")

In [ ]:
# Load the dataset
print("=" * 60)
print("STEP 1: LOADING DATASET")
print("=" * 60)

df = load_fraud_data()

print("\n📊 Initial Data Check:")
print(f"\t• Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\t• Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\t• Features: {df.shape[1] - 1} (excluding target)")
print("\t• Target: Class")


## Step 2: Schema Validation

**Expected Schema:**
- `Time`: Numeric (seconds elapsed)
- `V1-V28`: Numeric (PCA components, 28 features)
- `Amount`: Numeric (transaction amount)
- `Class`: Binary (0=legitimate, 1=fraud)

**Total**: 31 columns, all numeric

In [ ]:
# Schema validation
print("=" * 60)
print("STEP 2: SCHEMA VALIDATION")
print("=" * 60)

# Expected columns
expected_columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
expected_count = 31

print("\n📋 Column Validation:")
print(f"\t• Expected columns: {expected_count}")
print(f"\t• Actual columns: {len(df.columns)}")
print(f"\t• Match: {'✅ YES' if len(df.columns) == expected_count else '❌ NO'}")

# Check for missing or extra columns
missing_cols = set(expected_columns) - set(df.columns)
extra_cols = set(df.columns) - set(expected_columns)

if missing_cols:
    print(f"\n⚠️ Missing columns: {missing_cols}")
if extra_cols:
    print(f"\n⚠️ Extra columns: {extra_cols}")
if not missing_cols and not extra_cols:
    print("\n✅ All expected columns present")

# Display column names for verification
print(f"\n📝 Actual Columns: ({len(df.columns)}):")
print(f"\t{list(df.columns)[:5]} ... {list(df.columns)[-3:]}")

In [ ]:
# Data type validation
print("=" * 60)
print("DATA TYPE ANALYSIS")
print("=" * 60)

print("\n🔢 Data Types Summary:")
print(df.dtypes.value_counts())

print("\n📊 Detailed Data Types:")
print(f"\t• Time: {df['Time'].dtype}")
print(f"\t• V1-V28: {df[[f'V{i}' for i in range(1, 29)]].dtypes.unique()}")
print(f"\t• Amount: {df['Amount'].dtype}")
print(f"\t• Class: {df['Class'].dtype}")

# Check if all columns are numeric
all_numeric = df.select_dtypes(include=[np.number]).shape[1] == df.shape[1]
print(f"\n✅ All columns numeric: {'YES' if all_numeric else 'NO'}")

# Verify Class is binary (0 and 1 only)
unique_classes = df["Class"].unique()
print("\n🎯 Target Variable (Class):")
print(f"\t• Unique values: {sorted(unique_classes)}")
print(f"\t• Is binary: {'✅ YES' if len(unique_classes) == 2 else '❌ NO'}")
print(f"\t• Valid range [0, 1]: {'✅ YES' if set(unique_classes).issubset({0, 1}) else '❌ NO'}")

In [ ]:
# Display first few rows to visually inspect structure
print("\n" + "=" * 60)
print("SAMPLE DATA INSPECTION")
print("=" * 60)

print("\n📄 First 3 rows:")
print(df.head(3))

print("\n📄 Last 3 rows:")
print(df.tail(3))

print("\n📄 Random 3 rows:")
print(df.sample(3, random_state=42))

print("\n✅ Schema validation complete - All checks passed!")

## Step 3: Data Quality Assessment

**Quality Checks:**
1. Missing values analysis
2. Duplicate records detection
3. Statistical summary (5-number summary + mean/std)
4. Feature range validation
5. Cardinality check

In [ ]:
# Missing values analysis
print("=" * 60)
print("STEP 3: DATA QUALITY ASSESSMENT")
print("=" * 60)

print("\n🔍 MISSING VALUES ANALYSIS")
print("-" * 60)

# Check for missing values
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percentage": missing_pct
})

missing_df = missing_df[missing_df["Missing_Count"] > 0].sort_values("Missing_Count", ascending=False)

if len(missing_df) > 0:
    print(f"\n⚠️ Found missing values in {len(missing_df)} columns:")
    print(missing_df)
else:
    print("\n✅ No missing values detected in any column")
    print(f"\tTotal cells: {df.shape[0] * df.shape[1]:,}")
    print("\tAll cells populated: 100%")

# Check for any infinite values
print("\n🔍 INFINITE VALUES CHECK")
print("-" * 60)

inf_counts = np.isinf(df.select_dtypes(include=[np.number])).sum()
total_inf = inf_counts.sum()

if total_inf > 0:
    print(f"⚠️ Found {total_inf} infinite values:")
    print(inf_counts[inf_counts > 0])
else:
    print("✅ No infinite values detected")


In [ ]:
# Duplicate records analysis
print("\n" + "=" * 60)
print("DUPLICATE RECORDS ANALYSIS")
print("=" * 60)

# Check for complete duplicate rows
duplicate_rows = df.duplicated().sum()
duplicate_pct = (duplicate_rows / len(df)) * 100

print("\n📊 Complete Duplicate Rows:")
print(f"\t• Count: {duplicate_rows:,}")
print(f"\t• Percentage: {duplicate_pct:.4f}%")

if duplicate_rows > 0:
    print(f"\t⚠️ Warning: Found {duplicate_rows:,} duplicate transactions")
    print("\t  Consider: Investigation needed - are these legitimate repeated transactions?")
else:
    print("\t✅ No complete duplicate rows")

# Check for duplicate transactions based on Time + Amount (potential duplicates)
# This is important for fraud detection - same amount at same time could be suspicious
print("\n📊 Potential Duplicate Transactions (Time + Amount):")
time_amount_duplicates = df.duplicated(subset=['Time', 'Amount'], keep=False).sum()
print(f"\t• Records with same Time AND Amount: {time_amount_duplicates:,}")
print(f"\t• Percentage: {(time_amount_duplicates / len(df)) * 100:.2f}%")

if time_amount_duplicates > 0:
    print("\tℹ️ Note: These may be legitimate (e.g., multiple $10 transactions)")
    print("\t  Will investigate further in EDA phase")


In [ ]:
# Show all duplicated rows (including all occurrences)
duplicates_df = df[df.duplicated(keep=False)]
print(f"Found {len(duplicates_df)} duplicated rows")

# Display first few duplicates for inspection
print(duplicates_df.head(10))

In [ ]:
# Comprehensive statistical summary
print("\n" + "=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)

# Use our utility function for quick summary
quick_data_summary(df)

print("\n" + "=" * 60)
print("DETAILED STATISTICS (5-NUMBER SUMMARY)")
print("=" * 60)

# Get descriptive statistics
stats = df.describe()
print(stats)


In [ ]:
# Feature range analysis
print("\n" + "=" * 60)
print("FEATURE RANGE ANALYSIS")
print("=" * 60)

# Key features to examine
key_features = ["Time", "Amount", "Class"]
v_features = [f"V{i}" for i in range(1, 29)]

print("\n📊 Key Features:")
for col in key_features:
    print(f"\n {col}:")
    print(f"\tMin:      {df[col].min():>15,.2f}")
    print(f"\tMax:      {df[col].max():>15,.2f}")
    print(f"\tRange:    {df[col].max() - df[col].min():>15,.2f}")
    print(f"\tMean:     {df[col].mean():>15,.2f}")
    print(f"\tMedian:   {df[col].median():>15,.2f}")

# V features summary (PCA components)
print("\n📊 PCA Features (V1 - V28):")
v_df = df[v_features]
print(f"\tMin across all:       {v_df.min().min():>10.3f}")
print(f"\tMax across all:       {v_df.max().max():>10.3f}")
print(f"\tMean (should be ~0):  {v_df.mean().mean():>10.6f}")
print(f"\tStd (should be ~1):   {v_df.std().std():>10.3f}")

# Check if V features are standardized (expected for PCA)
print("\nℹ️ V features appear to be PCA-transformed:")
print(f"\t• Mean ~= 0: {'✅' if abs(v_df.mean().mean()) < 0.01 else '❌'}")
print(f"\t• Std ~= 1: {'✅' if 0.8 < v_df.std().mean() < 1.2 else '❌'}")


In [ ]:
# Final quality assessment summary
print("=" * 60)
print("DATA QUALITY ASSESSMENT SUMMARY")
print("=" * 60)

quality_checks = {
    "No missing values": missing_counts.sum() == 0,
    "No infinite values": total_inf == 0,
    "No complete duplicates": duplicate_rows == 0,
    "All columns numeric": df.select_dtypes(include=[np.number]).shape[1] == df.shape[1],
    "Binary target (0, 1)" : set(df["Class"].unique()).issubset({0, 1}),
    "V features standardized": abs(v_df.mean().mean()) < 0.1 and 0.8 < v_df.std().mean() < 1.2,
    "Expected row count (~284K)": 280000 < len(df) < 290000,
    "Expected column count (31)": len(df.columns) == 31,
}

print(f"\n✅ Quality Checks Passed: {sum(quality_checks.values())} / {len(quality_checks)}\n")

for check, passed in quality_checks.items():
    icon = "✅" if passed else "❌"
    print(f"\t{icon} {check}")

# Calculate overall quality score
quality_score = (sum(quality_checks.values()) / len(quality_checks)) * 100

print(f"\n📊 Overall Data Quality Score: {quality_score:.1f}%")

if quality_score == 100:
    print("\tExceellent! Dataset is high quality and ready for analysis")
elif quality_score >= 80:
    print("\tGood quality - minor issues to address")
else:
    print("\tQuality concerns detected - review required")

print("\n✅ Step 3 complete - Data quality validated!")


In [ ]:
# Detailed duplicate analysis
print("=" * 60)
print("STEP 4: DUPLICATE INVESTIGATION")
print("=" * 60)

# Find duplicate rows
duplicate_mask = df.duplicated(keep=False)
duplicate_records = df[duplicate_mask].copy()

print("\n📊 Duplicate Records Analysis:")
print("  • Total duplicate records: {len(duplicate_records):,}")
print("  • Unique transactions (group): {len(duplicate_records) // 2:,}")
print("  • Percentage of datasets: {len(duplicate_records)/ len(df) * 100:.2f}%")

# Check class distribution in duplicates
print("\n🎯 Class Distribution in Duplicates:")
dup_class_dist = duplicate_records["Class"].value_counts().sort_index()
for class_val, count in dup_class_dist.items():
    pct = (count/ len(duplicate_records)) * 100
    label = "Legitimate" if class_val == 0 else "Fraud"
    print(f"  Class {class_val} ({label}): {count:,} ({pct:.2f}%)")

# Compare to overall distribution
print("\n📊 Comparison to Overall Distribution:")
overall_fraud_pct = (df["Class"].sum() / len(df)) * 100
dup_fraud_pct = (duplicate_records["Class"].sum() / len(duplicate_records)) * 100

print("  • Fraud rate in full dataset: {overall_fraud_pct:.2f}%")
print("  • Fraud rate in duplicates: {dup_fraud_pct:.2f}%")

if dup_fraud_pct > overall_fraud_pct * 1.5:
    print("  ⚠️ Duplicates contain HIGHER fraud rate. Investage further")
elif dup_fraud_pct < overall_fraud_pct * 0.5:
    print(" ℹ️ Duplicates contain LOWER fraud rate. Mostly legitimate")
else:
    print("  ✅ Duplicates have similar fraud rate to overall data")

In [ ]:
# Examine sample duplicates
print("=" * 60)
print("SAMPLE DUPLICATE RECORDS")
print("=" * 60)

# Get first duplicate group
if len(duplicate_records) > 0:
    # Find first set of duplicates
    first_dup_group = df[df.duplicated(keep=False)].head(10)
    
    print("\n📋 Example Duplicate Group (Shoing first 10 records):")
    print(first_dup_group[["Time", "V1", "V2", "V3", "Amount", "Class"]])
    
    # Check if these are exact duplicates or just Time+Amount duplicates
    print("\n🔍 Duplicate Type Analysis:")
    print("  • Complete duplicate rows detected: All 31 features are identical")
    print("  • Not just Time + Amount matches")
    
    # Decision guidance
    print("\n💡 Recommendation:")
    print("  • Complete duplicate rows detected")
    print("  • Could be: Data collection artrifacts, system errors, or legitimate repeated transactions")
    print("  • Strategy: Keep one copy of each duplicate, remove others")
    print("  • This preserves information while avoiding data leakage")
    print("  • Will remove duplicates in Phase 3: Data Cleaning")
else:
    print("  No duplicates to display")

In [ ]:
# Class imbalance deep dive with visualization
print("=" * 60)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 60)

# Calculate detailed imbalance metrics
class_counts = df["Class"].value_counts().sort_index()
total = len(df)

print("\n📊 Detailed Class Distribution:")
print("  Class 0 (Legitimate):")
print(f"  • Count: {class_counts[0]:,}")
print(f"  • Percentagee: {(class_counts[0]/total)*100:.4f}%")
print(f"\n  Class 1 (Fraud):")
print(f"  • Count: {class_counts[1]:,}")
print(f"  • Percentage: {(class_counts[1])*100:.4f}%")

# Imbalance ratio
imbalance_ratio = class_counts[1] / class_counts[0]
ratio_inverse = class_counts[0] / class_counts[1]

print("\nImbalance Metrics:")
print(f"  • Imbalance Ratio: {imbalance_ratio:.6f}%")
print(f"  • Ratio (Majority : Minority): {ratio_inverse:.1f}:1")
print(f"  • For every 1 fraud: {ratio_inverse:.0f} legitimate transactions")

# Severity assessment
print("\n🚨 Imbalance Severity: EXTREME")
print("  • Classification: Highly imbalanced (500:1)")
print("  • Impact: Standard classifiers will predict all as legitimate")
print("  • Required techniques:")
print("    - ✓ Use PR-AUC instead of accuracy")
print("    - ✓ Consider SMOTE/ADASYN oversampling")
print("    - ✓ Apply class weights in models")
print("    - ✓ Use stratifier sampling for validation")
print("    - ✓ Adjust decision threshold for business objectives")

# Use utility function to visualize
print("\n📊 Visualize class distribution...")
plot_target_distribution(df)

In [ ]:
# Calculate baseline metrics (what we need to beat)
print("=" * 60)
print("BASELINE METRICS (NAIVE APPROACH)")
print("=" * 60)

print("\n🎯 If we predict all transactions as legitimate (Class 0):")
print(f"  • Accuracy: {class_counts[0]/total*100:.2f}%")
print("  • Precision: Not defined (no positive predictions)")
print("  • Recall: 0% (catch no frauds)")
print("  • F1-Score: 0%")

print("\n💡 Key Insight:")
print(f"  • Accuracy is useless as a metric ({class_counts[0]/total*100:.2f}% by doing nothing)")
print("  • Must use PR-AUC, ROC-AUC, and Recall@Precision metrics")
print("  • Business goal: Maximize fraud detection while minimizing false positives")

print("\n💰 Business Cost Considerations:")
print("  • False Negative (miss fraud): High cost")
print("    - Direct financial loss")
print("    - Regulatory penalties")
print("    - Reputational damage")
print("\n  • False Positive (block legitimate): High cost")
print("    - Customer dissatisfaction")
print("    - Lost transactions")
print("    - Support overhead")
print("\n  ⚖️ Balance: Need high recall (catch frauds) at acceptable precision")


In [ ]:
# Summary of findings
print("=" * 60)
print("STEP 4 SUMMARY - KEY FINDINGS")
print("=" * 60)

findings = {
    "Duplicates": {
        "status": "⚠️ ATTENTION NEEDED",
        "details": [
            "1081 complete duplicate rows found (0.38%)",
            "Will remove in Phase 3: Data Cleaning",
            "Strategy: Keep first occurrence, drop duplicates"
        ]
    },
    "Class Imbalance": {
        "status": "🔴 EXTREME",
        "details": {
            "Fraud rate: 0.17% (577.9:1 ratio)",
            "Requires specialized techniques (SMOTE, class weights)",
            "PR-AUC is primary metric, not accuracy"
        }
    },
    "Data Quality": {
        "status": "✅ EXCELLENT",
        "details": [
            "Zero missing values",
            "No infinite values",
            "PCA features properly standardized",
            "All data types valid"
        ]
    },
    "Feature Engineering Needs": {
        "status": "ℹ️ IDENTIFIED",
        "details": {
            "Time feature spans 48 hours (2 days)",
            "Amount ranges: $0 - $25,691",
            "V1-V28 are PCA components (anonymous)",
            "May need temporal features in Phase 4"
        }
    }
}

for category, info in findings.items():
    print(f"\n{info['status']} {category}:")
    for detail in info["details"]:
        print(f"• {detail}")

print("\nStep 4 complete - Duplicates identified, imbalance quantified")
print("\nNext: Great Expecattions validation suite")